In [4]:
import json

from src.runtime_constants import rr

rr.set_env(rr.ENV_PROD)
prompts_sheet = '1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI'
prompts_dms_guid = '38506c50-b1e4-4017-84f0-a2a52fa1465b'


In [ ]:
from src.db import DB

devs = [
'Kabam',
'Tripledot',
'Stillfront',
'Miniclip',
'King',
'Electronic Arts',
'Plarium',
'Tilting Point',
'Superplus Games',
'Exient Publishing',
'Carry 1st',
'Voodoo',
'Jam City',
'Wooga',
'My Games',
'Bandai Namco',
'AppLovin'
]



DB.commit()
DB.close()

In [3]:
google_play_scraper.process()

In [5]:
from src.server.models.dms_model import DmsModel
from src import external_api
from src.db import DbConnection


conn = DbConnection()
sheet_data = external_api.read_sheet('1P__GHUVGUcnmRTUA1L7uF4CNkZD2VN7FLty9tS-4ltI', 'Prompts', True)
DmsModel.save_json_to_dms(conn, sheet_data, prompts_dms_guid, 'gpt prompts')
conn.commit()
conn.close()

In [6]:
from src.db import DbConnection
from src.external_api import openai_api
import json
from src.server.models.dms_model import DmsModel

conn = DbConnection()
sheet_data = DmsModel.get_dms_data_to_json(conn, guid=prompts_dms_guid)
prompts = []
for it in sheet_data:
    if it['Agent'] == 'GPT4' and it['Type'] == 'Live':
        prompts.append({'area': it['Area'], 'prompt': it['Prompt']})

rows = conn.select_all("""
SELECT app_id, title FROM scraped_data.apps ORDER BY rank_val DESC LIMIT 10
""")

#print(f"Rows: {rows}")
#print(f"Prompts: {prompts}")

conn.close()

gpt_game = {}
for row in rows:
    gpt_result = []
    app_id = row['google_id']
    print(f"Title: {row['title']}")
    for p in prompts:
        msg = p['prompt'].replace("[GAME TITLE]", row['title'])
        gpt = openai_api.chat_gpt(msg)
        gpt_result.append({
            'area': p['area'],
            'result': gpt
        })
        #break
    conn = DbConnection()
    conn.query("""
    DELETE FROM tagged_data.prompts_queue WHERE app_id = %s
    """, [app_id])
    conn.query("""
    INSERT INTO tagged_data.prompts_queue (app_id, result) VALUES (%s, %s)
    """, [app_id, json.dumps(gpt_result)])
    conn.commit(True)

#gpt_result

Title: Candy Crush Saga
Title: 8 Ball Pool
Title: Candy Crush Soda Saga
Title: Plants vs. Zombies™
Title: FIFA Soccer
Title: Helix Jump
Title: Farm Heroes Saga
Title: Real Racing  3
Title: aquapark.io
Title: Plants vs Zombies™ 2


In [3]:
from src import external_api
from src.db import DbConnection

gpt_test_sheet = '1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys'

values = []
values.append(['id', 'title', 'area', 'response'])

conn = DbConnection()
rows = conn.select_all("""
SELECT g.app_id, g.title, p.result
  FROM tagged_data.prompts_queue p, scraped_data.apps g
 WHERE g.app_id = p.app_id
""")

for row in rows:
    result = json.loads(row['result'])
    for it in result:
        values.append([row['google_id'], row['title'], it['area'], it['result']])

external_api.write_sheet(gpt_test_sheet, 'Data', values)

{'spreadsheetId': '1TIu0NXPH8LFgVmlzA4-y2GdI80MbnQW_-FAvAxDCvys',
 'updatedRange': 'Data!A1:D28',
 'updatedRows': 28,
 'updatedColumns': 4,
 'updatedCells': 112}

In [27]:
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.common.by import By
from time import sleep
from src.utils.web import parse_web_page, get_web_page, get_driver
from selenium.webdriver.support.ui import WebDriverWait

driver = get_driver("https://chat.openai.com/auth/login")
#email_input = soup.find("input", attrs={"id": "username"})
#driver.find_element()
login_div: WebElement = driver.find_element(by=By.XPATH, value="//div[text()='Log in']")
login_btn: WebElement = login_div.find_element(by=By.XPATH, value="./..")
if login_btn is not None:
    #email_input.send_keys('pavel.dolezal@gembase.io')
    login_btn.click()
    #sleep(5)
    WebDriverWait(driver, 5)
#soup